In [1]:
import openai

In [2]:
client = openai.Client()

In [3]:
assistant = client.beta.assistants.create(
    name="Tutor de Tecnologia",
    instructions="Você é um tutor sobre assuntos relacionados a tecnologia",
    tools=[{"type":"code_interpreter"}],
    model="gpt-3.5-turbo-0125"
)

In [4]:
pergunta = """
 Em um sistema de cache com mapeamento direto, dado uma memória principal de uma cache
 de 256 KB e blocos de 64 bytes:
 - Quantas linhas existem na cache?
 - Onde o bloco de endereço 0x1A23C será mapeado na cache?
"""

In [5]:
# Criação da Thread
thread = client.beta.threads.create()
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content=pergunta
)

In [6]:
# Executa a thread
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
    instructions="Nome de usuário premium"
)

In [8]:
run.status

'queued'

In [9]:
# Aguarda a thread rodar
import time
while run.status in ["queued", "in_progress", "cancelling"]:
    time.sleep(1)
    run = client.beta.threads.runs.retrieve(
        thread_id=thread.id,
        run_id=run.id
    )

In [11]:
# Verifica a resposta
if run.status == "completed":
    mensagens = client.beta.threads.messages.list(
        thread_id=thread.id
    )
    print(mensagens)
else:
    print(f"Erro {run.status}")

SyncCursorPage[Message](data=[Message(id='msg_4Uwp4OlpsiuvnKNPRKtBmL8J', assistant_id='asst_ouDRQJCBRCFusRGVmS5urROZ', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='O bloco de endereço 0x1A23C será mapeado na linha 1672 da cache.\n\nSe precisar de mais alguma coisa, fico à disposição.'), type='text')], created_at=1736969078, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_pWs8qnGIs6NEgBQGX6q1pgBl', status=None, thread_id='thread_Pw5VdePKWikyTH3nqyWfWeBZ'), Message(id='msg_t6iSGJGX7jfqtvY2a8nhMkGO', assistant_id='asst_ouDRQJCBRCFusRGVmS5urROZ', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='A cache terá 4096 linhas.\n\nAgora, vamos determinar onde o bloco de endereço 0x1A23C será mapeado na cache. Para isso, iremos calcular o índice de mapeamento desse bloco na cache.'), type='text')], created_at=1736969075, incomplete_at=None, i

In [12]:
print(mensagens.data[0].content[0].text.value)

O bloco de endereço 0x1A23C será mapeado na linha 1672 da cache.

Se precisar de mais alguma coisa, fico à disposição.


In [13]:
# Analisando os passos do modelo
run_steps = client.beta.threads.runs.steps.list(
    thread_id=thread.id,
    run_id=run.id
)

In [14]:
for step in run_steps.data[::-1]:
    print(f"\n====Step {step.step_details.type}")
    if step.step_details.type == "tool_calls":
        for tool_call in step.step_details.tool_calls:
            print("=" *10)
            print(tool_call.code_interpreter.input)
            print("="*10)
    if step.step_details.type == "message_creation":
        message = client.beta.threads.messages.retrieve(
            thread_id=thread.id,
            message_id=step.step_details.message_creation.message_id
        )
        
        print(message.content[0].text.value)


====Step message_creation
Para determinar quantas linhas existem na cache em um sistema de mapeamento direto, precisamos dividir o tamanho total da cache pelo tamanho de cada bloco. 

Vamos realizar os cálculos para encontrar o número de linhas na cache e para mapear o bloco de endereço 0x1A23C na cache.

====Step tool_calls
# Tamanho total da cache em bytes
tamanho_cache_bytes = 256 * 1024  # 256 KB em bytes

# Tamanho de cada bloco em bytes
tamanho_bloco_bytes = 64

# Número de linhas na cache
num_linhas_cache = tamanho_cache_bytes // tamanho_bloco_bytes

num_linhas_cache

====Step message_creation
A cache terá 4096 linhas.

Agora, vamos determinar onde o bloco de endereço 0x1A23C será mapeado na cache. Para isso, iremos calcular o índice de mapeamento desse bloco na cache.

====Step tool_calls
# Endereço do bloco a ser mapeado
endereco_bloco = 0x1A23C

# Tamanho do bloco em bits (para determinar o offset)
tamanho_bloco_bits = 6  # Bloco de 64 bytes

# Número de linhas na cache (par